In [ ]:
!pip install faster-whisper yt-dlp ffmpeg-python flask flask-cors pyngrok pinecone sentence-transformers transformers torch groq -q
!apt-get install ffmpeg -y -q
!pip install -U yt-dlp

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 182.3/182.3 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 56.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 78.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 742.8/742.8 kB 31.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.7/141.7 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.4/36.4 MB 47.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.0/39.0 MB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 91.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 280.7/280.7 kB 26.8 MB/s eta 0:00:00
Reading package lists...
Building dependency tree...
Reading state information...
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 42 not upgraded.


In [ ]:
from pyngrok import ngrok

import os
os.environ["NGROK_AUTH_TOKEN"] = "Your ngrok api key"


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import subprocess, re, glob, wave, json, uuid, os, hashlib, shutil, sqlite3
import yt_dlp
from flask import Flask, request, jsonify, send_from_directory, Response, stream_with_context
from flask_cors import CORS
from pyngrok import ngrok
from collections import Counter
from groq import Groq
from faster_whisper import WhisperModel
from sentence_transformers import SentenceTransformer
from pinecone import Pinecone, ServerlessSpec
import time
import concurrent.futures
import threading
import numpy as np
import queue

# ══════════════════════════════════════════
# CONFIG & FOLDERS
# ══════════════════════════════════════════
FRONTEND_DIR   = "/content/drive/MyDrive/major/major1" # Update if needed
UPLOAD_DIR     = "/content/drive/MyDrive/major/vidmind"
DB_PATH        = "/content/drive/MyDrive/major/vidmind/vidmind.db"
# Coarse passages for Pinecone (semantic retrieval)
CHUNK_DURATION_COARSE = 120
CHUNK_OVERLAP_COARSE  = 25
MAX_CHUNK_LEN  = 30

os.makedirs(UPLOAD_DIR, exist_ok=True)
os.makedirs("speech_chunks", exist_ok=True)

app = Flask(__name__)
CORS(app)

# ══════════════════════════════════════════
# API KEYS & MODELS SETUP
# ══════════════════════════════════════════

# ── 3 Groq API keys: 2 for summary (round-robin), 1 dedicated for RAG ──
GROQ_API_KEY_SUMMARY_1 = os.environ.get(
    "GROQ_API_KEY_SUMMARY_1",
    "Your grok api key",   # ← key 1 (summary)
)
GROQ_API_KEY_SUMMARY_2 = os.environ.get(
    "GROQ_API_KEY_SUMMARY_2",
    "Your grok api key",   # ← key 2 (summary)
)
GROQ_API_KEY_RAG = os.environ.get(
    "GROQ_API_KEY_RAG",
    "Your grok api key",                           # ← key 3 (RAG/chat)
)

# Summary clients — alternated per part to avoid rate limits
_summary_clients = [
    Groq(api_key=GROQ_API_KEY_SUMMARY_1),
    Groq(api_key=GROQ_API_KEY_SUMMARY_2),
]
_summary_key_idx = 0
_summary_key_lock = threading.Lock()

def _get_summary_client():
    global _summary_key_idx
    with _summary_key_lock:
        client = _summary_clients[_summary_key_idx % len(_summary_clients)]
        _summary_key_idx += 1
        return client

# RAG client — dedicated key so summaries never exhaust it
groq_client = Groq(api_key=GROQ_API_KEY_RAG)
GROQ_MODEL  = "llama-3.3-70b-versatile"

# ── Load models in parallel threads so startup is faster ──────────
def _load_whisper_small():
    return WhisperModel("small", device="cuda", compute_type="float16")

def _load_whisper_large():
    for model_name in ("large-v3-turbo", "large-v3"):
        try:
            return WhisperModel(model_name, device="cuda", compute_type="float16")
        except Exception:
            print(f"{model_name} unavailable, trying next...")
    raise RuntimeError("No suitable Whisper model found.")

def _load_embed():
    # ✅ UPGRADED: BGE-large-en-v1.5 — significantly better retrieval accuracy
    m = SentenceTransformer("BAAI/bge-large-en-v1.5")
    m.max_seq_length = 512   # BGE-large benefits from full context
    return m

print("Loading models in parallel...")
with concurrent.futures.ThreadPoolExecutor(max_workers=3) as ex:
    f_small    = ex.submit(_load_whisper_small)
    f_large    = ex.submit(_load_whisper_large)
    f_embed    = ex.submit(_load_embed)
    detect_model     = f_small.result()
    transcribe_model = f_large.result()
    embed_model      = f_embed.result()

# Warm up the transcription model
print("Warming up Whisper...")
_dummy = np.zeros(16000, dtype=np.float32)
list(transcribe_model.transcribe(_dummy, beam_size=1)[0])
print("Whisper warm-up done.")

# ── Pinecone ───────────────────────────────────────────────────────
PINECONE_API_KEY = "Your Pincone api key"
INDEX_NAME       = os.environ.get("PINECONE_INDEX_NAME", "video-transcript-index-v7")

print("Connecting to Pinecone...")
pc = Pinecone(api_key=PINECONE_API_KEY)

def _init_pinecone_index():
    existing = pc.list_indexes().names()
    if INDEX_NAME not in existing:
        print(f"Index '{INDEX_NAME}' not found — creating...")
        pc.create_index(
            name=INDEX_NAME,
            dimension=1024,     # ✅ BGE-large outputs 1024-dim vectors (was 768)
            metric="cosine",
            spec=ServerlessSpec(cloud="aws", region="us-east-1"),
        )
        for attempt in range(60):
            status = pc.describe_index(INDEX_NAME).status
            if status.get("ready"):
                print(f"✅ Index '{INDEX_NAME}' is ready.")
                break
            print(f"  waiting for index... ({attempt + 1})")
            time.sleep(2)
        else:
            raise RuntimeError(f"Pinecone index '{INDEX_NAME}' did not become ready in time.")
    else:
        print(f"✅ Index '{INDEX_NAME}' already exists.")
    return pc.Index(INDEX_NAME)

index = _init_pinecone_index()

print("🚀 All models and databases ready!")

# ══════════════════════════════════════════
# SQLITE DATABASE SETUP
# ══════════════════════════════════════════
_db_lock = threading.Lock()

def init_db():
    conn = sqlite3.connect(DB_PATH)
    c = conn.cursor()
    c.execute("""CREATE TABLE IF NOT EXISTS videos
                 (vid_hash TEXT PRIMARY KEY, url TEXT, summary TEXT,
                  language TEXT, chunks TEXT, title TEXT, thumbnail_url TEXT)""")
    for col in ("title", "thumbnail_url"):
        try:
            c.execute(f"ALTER TABLE videos ADD COLUMN {col} TEXT")
        except sqlite3.OperationalError:
            pass
    conn.commit()
    conn.close()

init_db()

def get_video_from_db(vid_hash):
    with _db_lock:
        conn = sqlite3.connect(DB_PATH)
        c = conn.cursor()
        c.execute(
            "SELECT url, summary, language, chunks, title, thumbnail_url FROM videos WHERE vid_hash=?",
            (vid_hash,),
        )
        row = c.fetchone()
        conn.close()
    if row:
        return {
            "url": row[0],
            "summary": row[1],
            "language": row[2],
            "chunks": json.loads(row[3]) if row[3] else [],
            "title": row[4] or "",
            "thumbnail_url": row[5] or "",
        }
    return None

def save_video_to_db(vid_hash, url, summary, language, chunks, title="", thumbnail_url=""):
    with _db_lock:
        conn = sqlite3.connect(DB_PATH)
        conn.execute(
            "INSERT OR REPLACE INTO videos (vid_hash, url, summary, language, chunks, title, thumbnail_url) VALUES (?,?,?,?,?,?,?)",
            (vid_hash, url, summary, language, json.dumps(chunks), title or "", thumbnail_url or ""),
        )
        conn.commit()
        conn.close()

def list_videos_from_db():
    with _db_lock:
        conn = sqlite3.connect(DB_PATH)
        c = conn.cursor()
        c.execute(
            """SELECT vid_hash, url, language, title, thumbnail_url, summary FROM videos
               WHERE chunks IS NOT NULL AND length(trim(chunks)) > 2
               ORDER BY rowid DESC"""
        )
        rows = c.fetchall()
        conn.close()
    out = []
    for vid_hash, url, language, title, thumbnail_url, summary in rows:
        out.append(
            {
                "vid_hash": vid_hash,
                "url": url or "",
                "language": language or "",
                "title": title or "",
                "thumbnail_url": thumbnail_url or "",
                "summary_preview": (summary or "")[:280],
            }
        )
    return out

# ══════════════════════════════════════════
# PROGRESS EVENT QUEUES
# ══════════════════════════════════════════
# Maps vid_hash -> queue of SSE events
_progress_queues: dict[str, queue.Queue] = {}
_progress_lock = threading.Lock()

def _get_progress_queue(vid_hash: str) -> queue.Queue:
    with _progress_lock:
        if vid_hash not in _progress_queues:
            _progress_queues[vid_hash] = queue.Queue(maxsize=200)
        return _progress_queues[vid_hash]

def _push_progress(vid_hash: str, stage: str, pct: int, msg: str, extra: dict | None = None):
    """Push a progress event onto the queue for this video."""
    payload = {"stage": stage, "pct": pct, "msg": msg}
    if extra:
        payload.update(extra)
    try:
        q = _get_progress_queue(vid_hash)
        q.put_nowait(payload)
    except queue.Full:
        pass  # drop if no consumer

def _push_done(vid_hash: str, data: dict):
    _push_progress(vid_hash, "done", 100, "Complete", extra=data)

def _push_error(vid_hash: str, msg: str):
    _push_progress(vid_hash, "error", 0, msg)

# ══════════════════════════════════════════
# UTILS & GROQ HELPER
# ══════════════════════════════════════════
def make_url_hash(url):
    return hashlib.sha256(url.strip().split("&")[0].encode()).hexdigest()[:16]

def make_file_hash(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        h.update(f.read(4 * 1024 * 1024))
    return h.hexdigest()[:16]

def groq_chat(system, user, max_tokens=512, temperature=0.3):
    """RAG/general chat — uses the dedicated RAG key."""
    resp = groq_client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[{"role": "system", "content": system},
                  {"role": "user",   "content": user}],
        max_tokens=max_tokens,
        temperature=temperature,
    )
    return resp.choices[0].message.content.strip()

def groq_chat_summary(system, user, max_tokens=800, temperature=0.3):
    """Summary generation — alternates between the 2 summary keys."""
    client = _get_summary_client()
    resp = client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[{"role": "system", "content": system},
                  {"role": "user",   "content": user}],
        max_tokens=max_tokens,
        temperature=temperature,
    )
    return resp.choices[0].message.content.strip()

# ══════════════════════════════════════════
# AUDIO HELPERS
# ══════════════════════════════════════════
def detect_language(file):
    _, info = detect_model.transcribe(file, beam_size=1)
    return info.language

def get_audio_duration(path):
    result = subprocess.run(
        ["ffprobe", "-v", "error", "-show_entries", "format=duration",
         "-of", "default=noprint_wrappers=1:nokey=1", path],
        stdout=subprocess.PIPE, stderr=subprocess.DEVNULL, text=True
    )
    try:
        return float(result.stdout.strip())
    except ValueError:
        return 0.0

def split_audio_on_silence(audio_file):
    cmd = ["ffmpeg", "-i", audio_file,
           "-af", f"silencedetect=n=-35dB:d=0.7",
           "-f", "null", "-"]
    log = subprocess.run(cmd, stderr=subprocess.PIPE, text=True).stderr
    silence_ends = [
        float(re.search(r"silence_end: (\d+\.\d+)", line).group(1))
        for line in log.split("\n")
        if "silence_end" in line and re.search(r"silence_end: (\d+\.\d+)", line)
    ]
    chunks, start = [], 0.0
    for end in silence_ends:
        if end - start >= MAX_CHUNK_LEN:
            chunks.append((start, end))
            start = end
    chunks.append((start, None))
    return chunks

def _export_chunk(args):
    audio_file, i, s, e, out_dir = args
    cmd = ["ffmpeg", "-y", "-i", audio_file, "-ss", str(s)]
    if e:
        cmd += ["-to", str(e)]
    cmd += ["-ac", "1", "-ar", "16000", f"{out_dir}/chunk_{i:04d}.wav"]
    subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

def split_wav_into_files(audio_file, chunks, out_dir="speech_chunks"):
    shutil.rmtree(out_dir, ignore_errors=True)
    os.makedirs(out_dir, exist_ok=True)
    args = [(audio_file, i, s, e, out_dir) for i, (s, e) in enumerate(chunks)]
    with concurrent.futures.ThreadPoolExecutor(max_workers=8) as ex:
        list(ex.map(_export_chunk, args))

# ══════════════════════════════════════════
# TRANSCRIPTION  — ACCURACY OPTIMIZED
# ══════════════════════════════════════════
_whisper_lock = threading.Lock()

def _load_audio_as_array(path: str) -> tuple[np.ndarray, float]:
    with wave.open(path, "r") as w:
        frames = w.readframes(w.getnframes())
        dur    = w.getnframes() / w.getframerate()
    audio = np.frombuffer(frames, dtype=np.int16).astype(np.float32) / 32768.0
    return audio, dur

def transcribe_chunks(final_language, chunk_dir="speech_chunks", vid_hash=None):
    """Transcribe all chunks and push progress events.

    After the first 2 chunks are done we kick off an early partial summary
    so the frontend can show it while the rest transcribes.
    """
    chunk_files = sorted(glob.glob(f"{chunk_dir}/*.wav"))
    if not chunk_files:
        return []

    arrays, offsets, t = [], [], 0.0
    for f in chunk_files:
        audio, dur = _load_audio_as_array(f)
        arrays.append(audio)
        offsets.append(t)
        t += dur

    INITIAL_PROMPT = (
        "The following is an accurate transcript of a lecture, talk, or video. "
        "Punctuation and capitalization are correct."
    )

    full_transcript = []
    total = len(arrays)

    with _whisper_lock:
        for i, (audio, offset) in enumerate(zip(arrays, offsets)):
            segments_out, _ = transcribe_model.transcribe(
                audio,
                language=final_language,
                beam_size=5,
                best_of=5,
                temperature=[0.0, 0.2, 0.4, 0.6, 0.8, 1.0],
                condition_on_previous_text=True,
                initial_prompt=INITIAL_PROMPT,
                vad_filter=True,
                vad_parameters={"min_silence_duration_ms": 400},
                word_timestamps=True,
                without_timestamps=False,
            )
            for seg in segments_out:
                full_transcript.append(
                    (seg.start + offset, seg.end + offset, seg.text)
                )

            # Push progress after each chunk
            pct = 30 + int(50 * (i + 1) / total)   # 30–80% range
            if vid_hash:
                _push_progress(
                    vid_hash, "transcribing", pct,
                    f"Transcribing chunk {i + 1}/{total}…"
                )

            # After 2 chunks — push partial transcript for early summary
            if vid_hash and i == 1 and len(full_transcript) > 0:
                _push_progress(
                    vid_hash, "partial_transcript", pct,
                    "First chunks ready — starting early summary…",
                    extra={"partial": [
                        {"start": float(s), "end": float(e), "text": txt}
                        for s, e, txt in full_transcript
                    ]}
                )

    full_transcript.sort(key=lambda x: x[0])
    return full_transcript

# ══════════════════════════════════════════
# LANGUAGE DETECTION — MORE SAMPLES
# ══════════════════════════════════════════
def detect_language_parallel(audio_path):
    samples_dir = os.path.join(UPLOAD_DIR, "lang_samples")
    os.makedirs(samples_dir, exist_ok=True)

    total_dur = get_audio_duration(audio_path)
    seek_positions = [
        0,
        total_dur * 0.2,
        total_dur * 0.4,
        total_dur * 0.6,
        total_dur * 0.8,
    ]

    def _extract(args):
        ss, out = args
        subprocess.run(
            ["ffmpeg", "-y", "-i", audio_path, "-ss", str(ss),
             "-t", "30",
             "-ac", "1", "-ar", "16000", out],
            stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
        )
        return out

    sample_paths = [(ss, f"{samples_dir}/s{i}.wav")
                    for i, ss in enumerate(seek_positions)]

    with concurrent.futures.ThreadPoolExecutor(max_workers=5) as ex:
        outputs = list(ex.map(_extract, sample_paths))

    detected = []
    for out in outputs:
        if os.path.exists(out):
            detected.append(detect_language(out))

    return Counter(detected).most_common(1)[0][0] if detected else "en"

# ══════════════════════════════════════════
# EMBEDDINGS — BATCHED  (BGE-large)
# ══════════════════════════════════════════
def embed_batch(texts: list[str]) -> list[list[float]]:
    vecs = embed_model.encode(
        texts,
        batch_size=32,
        normalize_embeddings=True,
        show_progress_bar=False,
        convert_to_numpy=True,
    )
    return vecs.tolist()

def embed_query(query_text: str) -> list[float]:
    instruction = "Represent this sentence for searching relevant passages: "
    vec = embed_model.encode(
        instruction + query_text,
        normalize_embeddings=True,
        convert_to_numpy=True,
    )
    return vec.tolist()

# ══════════════════════════════════════════
# PASSAGE MERGING WITH OVERLAP
# ══════════════════════════════════════════
def merge_into_passages(full_transcript, duration_sec, overlap_sec):
    """Merge Whisper segment tuples (start, end, text) into overlapping windows for embeddings."""
    if not full_transcript:
        return []

    merged = []
    segments = full_transcript
    i = 0

    while i < len(segments):
        chunk_start = segments[i][0]
        chunk_texts = []
        chunk_end   = chunk_start + duration_sec
        j = i

        while j < len(segments) and segments[j][0] < chunk_end:
            chunk_texts.append(segments[j][2])
            j += 1

        if chunk_texts:
            end_time = segments[j - 1][1] if j > 0 else chunk_end
            merged.append((chunk_start, end_time, " ".join(chunk_texts)))

        next_start = chunk_start + duration_sec - overlap_sec
        while i < len(segments) and segments[i][0] < next_start:
            i += 1

        if i >= j and j < len(segments):
            i = j

    return merged

def upsert_to_pinecone(merged, vid_hash):
    texts  = [t for _, _, t in merged]
    embeds = embed_batch(texts)

    vectors = [
        {
            "id":     f"chunk_{i}",
            "values": embeds[i],
            "metadata": {
                "text":  texts[i],
                "start": float(s),
                "end":   float(e),
            },
        }
        for i, (s, e, _) in enumerate(merged)
    ]

    batches = [vectors[i:i+100] for i in range(0, len(vectors), 100)]
    with concurrent.futures.ThreadPoolExecutor(max_workers=4) as ex:
        futs = [ex.submit(index.upsert, vectors=b, namespace=vid_hash) for b in batches]
        for f in futs:
            f.result()
    print(f"✅ Upserted {len(vectors)} vectors to namespace: {vid_hash}")

# ══════════════════════════════════════════
# SUMMARY — BETTER COVERAGE (uses summary keys)
# ══════════════════════════════════════════
def generate_summary(merged_passages, chars_per_part=8000, vid_hash=None, part_offset=0):
    """Generate a rolling summary.

    vid_hash + part_offset allow pushing intermediate summary SSE events
    so the frontend can render partial summaries while transcription continues.
    """
    all_text = " ".join(t for _, _, t in merged_passages)

    parts = []
    for i in range(0, len(all_text), chars_per_part):
        part = all_text[i:i + chars_per_part].strip()
        if part:
            parts.append(part)

    if not parts:
        return "No transcript available."

    print(f"[Summary] {len(parts)} parts to summarize...")

    running_summary = ""

    for i, part in enumerate(parts):
        part_num   = i + 1
        total_parts = len(parts)

        if running_summary:
            user_msg = (
                f"Previous summary (parts 1–{i}):\n{running_summary}\n\n"
                f"New transcript part ({part_num}/{total_parts}):\n{part}"
            )
            system_msg = (
                f"You are summarizing a video transcript progressively.\n"
                f"You have a running summary of the first {i} part(s), "
                f"and now a new transcript chunk (part {part_num} of {total_parts}).\n"
                f"Produce an UPDATED combined summary with these sections:\n"
                f"**Overview** — 2-3 sentences about the whole video so far\n"
                f"**Key Topics** — updated bullet list\n"
                f"**Main Points** — concise paragraphs per topic\n"
                f"**Key Takeaway** — 1 sentence\n\n"
                f"Use ONLY information from the transcript and previous summary."
            )
        else:
            user_msg = (
                f"Transcript part (1/{total_parts}):\n{part}"
            )
            system_msg = (
                f"You are summarizing part 1 of {total_parts} of a video transcript.\n"
                f"Produce a structured summary with these sections:\n"
                f"**Overview** — 2-3 sentences\n"
                f"**Key Topics** — bullet list\n"
                f"**Main Points** — concise paragraphs per topic\n"
                f"**Key Takeaway** — 1 sentence\n\n"
                f"Use ONLY information from the transcript provided."
            )

        print(f"[Summary] Calling Groq (summary key) for part {part_num}/{total_parts}...")
        running_summary = groq_chat_summary(
            system_msg,
            user_msg,
            max_tokens=800,
            temperature=0.3,
        )
        print(f"[Summary] Part {part_num} done.")

        # Push intermediate summary to SSE so frontend can render it live
        if vid_hash:
            _push_progress(
                vid_hash, "partial_summary",
                80 + int(15 * (part_offset + part_num) / max(total_parts, 1)),
                f"Summary part {part_num}/{total_parts} ready",
                extra={"partial_summary": running_summary}
            )

    print("[Summary] ✅ Final summary ready.")
    return running_summary

# ══════════════════════════════════════════
# MAIN PROCESS PIPELINE
# ══════════════════════════════════════════
def process_audio(audio_path, vid_hash, youtube_url="", title="", thumbnail_url=""):
    """Full pipeline with SSE progress pushes."""

    _push_progress(vid_hash, "detecting_language", 5, "Detecting language & splitting audio…")

    # 1. Language detection + silence split concurrently
    with concurrent.futures.ThreadPoolExecutor(max_workers=2) as ex:
        lang_fut    = ex.submit(detect_language_parallel, audio_path)
        silence_fut = ex.submit(split_audio_on_silence, audio_path)
        final_language = lang_fut.result()
        silence_chunks = silence_fut.result()

    _push_progress(vid_hash, "exporting_chunks", 20, f"Language: {final_language} · exporting audio chunks…")

    # 2. Export wav chunks (parallel ffmpeg)
    split_wav_into_files(audio_path, silence_chunks)

    _push_progress(vid_hash, "transcribing", 30, "Transcribing audio with Whisper…")

    # 3. Transcribe (accuracy-optimized) with per-chunk progress
    full_transcript = transcribe_chunks(final_language, vid_hash=vid_hash)

    # Whisper segment-level rows for UI + RAG fine context
    fine_chunks = [
        {"start": float(s), "end": float(e), "text": (t or "").strip()}
        for s, e, t in full_transcript
    ]
    merged_coarse = merge_into_passages(
        full_transcript, CHUNK_DURATION_COARSE, CHUNK_OVERLAP_COARSE
    )

    _push_progress(vid_hash, "summarizing", 80, "Generating summary & indexing to Pinecone…")

    # 4. Summary + Pinecone upsert concurrently (coarse passages only)
    with concurrent.futures.ThreadPoolExecutor(max_workers=2) as ex:
        summary_fut = ex.submit(generate_summary, merged_coarse, 8000, vid_hash)
        upsert_fut  = ex.submit(upsert_to_pinecone, merged_coarse, vid_hash)
        summary = summary_fut.result()
        upsert_fut.result()

    _push_progress(vid_hash, "saving", 95, "Saving to database…")

    save_video_to_db(
        vid_hash,
        youtube_url,
        summary,
        final_language,
        fine_chunks,
        title=title,
        thumbnail_url=thumbnail_url,
    )

    return {"language": final_language, "summary": summary}

# ══════════════════════════════════════════
# CHAT & RAG LOGIC — PINECONE ONLY
# ══════════════════════════════════════════
def classify_question(q):
    q = q.lower().strip()
    if any(re.match(p, q) for p in [
        r"^(hi|hello|hey|sup)[\s!?.]*$",
        r"^(thanks?|ty|bye)[\s!?.]*$",
    ]):
        return "generic"
    if any(t in q for t in ["summarize", "summary", "overview", "tldr"]):
        return "summary"
    return "rag"

def generate_hypothetical_answer(question, summary=""):
    """HyDE: hypothetical passage for retrieval, conditioned on video summary when available."""
    summary = (summary or "").strip()
    if len(summary) > 4000:
        summary = summary[:4000] + "…"
    system = (
        "Write a short hypothetical passage (3-4 sentences) that a speaker in the video might say, "
        "such that it would help retrieve the right part of the transcript to answer the user's question. "
        " Output only the passage, no preamble."
    )
    user = (
        f"Video summary (what the video is about):\n{summary}\n\n"
        f"User question:\n{question}"
    )
    try:
        return groq_chat(system, user, max_tokens=180)
    except Exception:
        return question

def _fmt_span_ts(start_s, end_s):
    return f"[{start_s:.2f}s–{end_s:.2f}s]"


def _opening_segments(fine_segments, max_start=5.0, max_count=3):
    opening = [s for s in fine_segments if s["start"] < max_start][:max_count]
    if not opening and fine_segments:
        opening = fine_segments[: min(2, len(fine_segments))]
    return opening


def _select_fine_for_coarse(fine_segments, top_coarse, buffer_sec=30):
    selected, seen = [], set()
    for m in top_coarse[:7]:
        ws = max(0.0, float(m["metadata"]["start"]) - buffer_sec)
        we = float(m["metadata"]["end"]) + buffer_sec
        for seg in fine_segments:
            if seg["end"] <= ws or seg["start"] >= we:
                continue
            key = (round(seg["start"], 2), round(seg["end"], 2))
            if key in seen:
                continue
            seen.add(key)
            selected.append(seg)
    selected.sort(key=lambda x: x["start"])
    return selected  # no cap


def _parse_llm_json(text):
    t = (text or "").strip()
    if t.startswith("```"):
        t = re.sub(r"^```(?:json)?\s*", "", t, flags=re.IGNORECASE)
        t = re.sub(r"\s*```\s*$", "", t)
    return json.loads(t)


def answer_via_rag(question, vid_hash, youtube_url, summary, fine_segments, use_hyde=True, debug=False):
    steps = []
    summary_excerpt = (summary or "")[:1200]

    if use_hyde:
        hyde_text = generate_hypothetical_answer(question, summary)
        query_text = hyde_text
        if debug:
            steps.append(
                {
                    "step": len(steps) + 1,
                    "name": "HyDE",
                    "input": {
                        "question": question,
                        "summary_excerpt": summary_excerpt,
                    },
                    "output": hyde_text,
                }
            )
    else:
        query_text = question
        if debug:
            steps.append(
                {
                    "step": len(steps) + 1,
                    "name": "HyDE",
                    "skipped": True,
                    "embedding_query_text": question,
                }
            )

    query_vec = embed_query(query_text)
    if debug:
        steps.append(
            {
                "step": len(steps) + 1,
                "name": "Query embedding",
                "output": {
                    "model": "BAAI/bge-large-en-v1.5",
                    "dimension": len(query_vec),
                    "query_text_used": query_text[:2000],
                },
            }
        )

    raw_results = index.query(
        vector=query_vec,
        top_k=12,
        include_metadata=True,
        namespace=vid_hash,
    )["matches"]

    if debug:
        steps.append(
            {
                "step": len(steps) + 1,
                "name": "Pinecone vector query",
                "input": {"namespace": vid_hash, "top_k": 12},
                "output": {
                    "match_count": len(raw_results),
                    "matches": [
                        {
                            "id": m.get("id"),
                            "pinecone_score": float(m.get("score", 0) or 0),
                            "start": m["metadata"].get("start"),
                            "end": m["metadata"].get("end"),
                            "text_preview": (m["metadata"].get("text") or "")[:220],
                        }
                        for m in raw_results
                    ],
                },
            }
        )

    if not raw_results:
        err = {"error": "No relevant passages found."}
        if debug:
            err["debug"] = {"steps": steps + [{"step": len(steps) + 1, "name": "Abort", "reason": "no Pinecone matches"}]}
        return err

    top_coarse = sorted(raw_results, key=lambda x: x["score"], reverse=True)[:7]

    best_match = top_coarse[0]
    fallback_ts = int(max(0, float(best_match["metadata"]["start"]) - 2.0))

    if debug:
        steps.append(
            {
                "step": len(steps) + 1,
                "name": "Top coarse windows (for expansion)",
                "output": [
                    {
                        "start": float(m["metadata"]["start"]),
                        "end": float(m["metadata"]["end"]),
                        "text_preview": (m["metadata"].get("text") or "")[:200],
                    }
                    for m in top_coarse
                ],
            }
        )

    opening = _opening_segments(fine_segments)
    candidates = _select_fine_for_coarse(fine_segments, top_coarse)

    if debug:
        steps.append(
            {
                "step": len(steps) + 1,
                "name": "Whisper segments — opening (prepended to prompt)",
                "output": opening,
            }
        )
        steps.append(
            {
                "step": len(steps) + 1,
                "name": "Whisper segments — candidates (max 7, overlap coarse)",
                "output": candidates,
            }
        )

    lines = ["=== Opening of the video (first moments) ==="]
    for seg in opening:
        lines.append(f"{_fmt_span_ts(seg['start'], seg['end'])} {seg['text']}")
    lines.append("=== Candidate transcript snippets ===")
    for seg in candidates:
        lines.append(f"{_fmt_span_ts(seg['start'], seg['end'])} {seg['text']}")
    context_block = "\n".join(lines)

    system = (
        "You are VidMind AI. Answer or explain only using the timestamped transcript snippets above.\n"
        "Return a single JSON object (no markdown code fences), with exactly these keys:\n"
        '- "answer" (string, the user-facing reply)\n'
        '- "best_start_seconds" (number, MUST be plain seconds as a number e.g. 1475.62, '
        'NOT in MM:SS format. Convert [MM:SS] timestamps to seconds before returning.)\n'
        '- "confidence_note" (string, optional, may be empty)\n'
        '- "snippet_ranges" (array of objects with "start" and "end" as plain seconds numbers, '
        'NOT MM:SS format. Convert all timestamps to seconds.)\n'
        'If the snippets do not support an answer, say so in "answer" and use best_start_seconds 0.\n\n'
        "EXAMPLE OUTPUT (follow this exact format):\n"
        '{\n'
        '  "answer": "Unsupervised learning is when data is unlabeled and we try to find patterns.",\n'
        '  "best_start_seconds": 1475.62,\n'
        '  "confidence_note": "",\n'
        '  "snippet_ranges": [\n'
        '    {"start": 1475.62, "end": 1478.0},\n'
        '    {"start": 1478.0,  "end": 1480.5}\n'
        '  ]\n'
        '}\n\n'
        "NOTE: Timestamps in the transcript are already plain seconds, e.g. [2526.38s–2529.56s]. "
        "Use them directly as numbers — no conversion needed."
    )
    user_msg = f"{context_block}\n\nQuestion: {question}"

    if debug:
        steps.append({"step": len(steps) + 1, "name": "Final LLM — system prompt", "output": system})
        steps.append({"step": len(steps) + 1, "name": "Final LLM — user message (full)", "output": user_msg})

    answer_raw = groq_chat(system, user_msg, max_tokens=700, temperature=0.2)

    if debug:
        steps.append({"step": len(steps) + 1, "name": "Final LLM — raw response", "output": answer_raw})

    try:
        parsed = _parse_llm_json(answer_raw)
    except json.JSONDecodeError:
        try:
            fix = groq_chat(
                "Output ONLY valid minified JSON with keys answer, best_start_seconds, confidence_note, snippet_ranges.",
                f"Turn this into valid JSON only:\n{answer_raw[:2500]}",
                max_tokens=500,
                temperature=0.1,
            )
            parsed = _parse_llm_json(fix)
            if debug:
                steps.append({"step": len(steps) + 1, "name": "JSON repair attempt", "output": fix})
        except (json.JSONDecodeError, Exception) as e2:
            parsed = {
                "answer": answer_raw,
                "best_start_seconds": fallback_ts,
                "confidence_note": "",
                "snippet_ranges": [],
            }
            if debug:
                steps.append({
                    "step": len(steps) + 1,
                    "name": "JSON parse failed — fallback",
                    "output": {"error": str(e2), "fallback_timestamp": fallback_ts},
                })

    if debug:
        steps.append({"step": len(steps) + 1, "name": "Parsed JSON (client-facing)", "output": parsed})

    best_start = parsed.get("best_start_seconds", fallback_ts)
    try:
        best_start_i = int(max(0, round(float(best_start))))
    except (TypeError, ValueError):
        best_start_i = fallback_ts

    out = {
        "answer": str(parsed.get("answer", "") or answer_raw),
        "source": "rag",
        "confidence": float(best_match.get("score", 0) or 0),
        "timestamp_s": best_start_i,
        "timestamp_label": str(best_start_i),
        "youtube_url": youtube_url or "",
        "rag_meta": {
            "best_start_seconds": best_start_i,
            "snippet_ranges": parsed.get("snippet_ranges") or [],
            "confidence_note": (parsed.get("confidence_note") or "").strip(),
        },
    }
    if debug:
        out["debug"] = {"steps": steps}
    return out

# ══════════════════════════════════════════
# FLASK ROUTES
# ══════════════════════════════════════════
ALLOWED_EXTENSIONS = {"mp4", "mkv", "webm", "avi", "mov", "mp3", "wav", "m4a", "ogg"}

def _allowed_file(filename):
    return "." in filename and filename.rsplit(".", 1)[-1].lower() in ALLOWED_EXTENSIONS


def _index_html_name():
    for name in ("home.html", "landing .html", "landing.html"):
        if os.path.isfile(os.path.join(FRONTEND_DIR, name)):
            return name
    return "home.html"


@app.route("/")
def serve_index():
    return send_from_directory(FRONTEND_DIR, _index_html_name())


# ── SSE progress stream ───────────────────────────────
@app.route("/progress/<vid_hash>")
def progress_stream(vid_hash):
    """Server-Sent Events endpoint — client subscribes before triggering /load-youtube or /load-file."""
    q = _get_progress_queue(vid_hash)

    def generate():
        while True:
            try:
                payload = q.get(timeout=60)
                data = json.dumps(payload)
                yield f"data: {data}\n\n"
                if payload.get("stage") in ("done", "error"):
                    break
            except queue.Empty:
                yield ": keepalive\n\n"

    return Response(
        stream_with_context(generate()),
        mimetype="text/event-stream",
        headers={
            "Cache-Control": "no-cache",
            "X-Accel-Buffering": "no",
        },
    )


@app.route("/load-youtube", methods=["POST"])
def load_youtube():
    data = request.get_json(silent=True) or {}
    url  = data.get("url", "").strip()
    if not url:
        return jsonify({"error": "No URL provided"}), 400

    vid_hash  = make_url_hash(url)
    db_record = get_video_from_db(vid_hash)
    if db_record and db_record.get("chunks"):
        return jsonify(
            {
                "vid_hash": vid_hash,
                "language": db_record["language"],
                "summary": db_record["summary"],
                "title": db_record.get("title") or "",
                "thumbnail_url": db_record.get("thumbnail_url") or "",
                "from_db": True,
            }
        )

    audio_path = os.path.join(UPLOAD_DIR, "audio.wav")
    if os.path.exists(audio_path):
        os.remove(audio_path)

    _push_progress(vid_hash, "downloading", 3, "Downloading audio from YouTube…")

    ydl_opts = {
        "format": "bestaudio/best",
        "outtmpl": audio_path.replace(".wav", ".%(ext)s"),
        "postprocessors": [{"key": "FFmpegExtractAudio", "preferredcodec": "wav"}],
        "quiet": True,
        "writethumbnail": False,
        "writesubtitles": False,
        "writedescription": False,
        "no_warnings": True,
    }
    info = None
    try:
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            info = ydl.extract_info(url, download=True)
    except Exception as e:
        _push_error(vid_hash, f"Download failed: {e}")
        return jsonify({"error": f"Download failed: {e}"}), 500

    title = (info or {}).get("title") or ""
    thumbnail_url = (info or {}).get("thumbnail") or ""
    thumbs = (info or {}).get("thumbnails") or []
    if not thumbnail_url and thumbs:
        thumbnail_url = (thumbs[-1] or {}).get("url") or ""

    def _run():
        try:
            result = process_audio(
                audio_path,
                vid_hash,
                youtube_url=url,
                title=title,
                thumbnail_url=thumbnail_url,
            )
            _push_done(vid_hash, {
                "vid_hash": vid_hash,
                "language": result["language"],
                "summary": result["summary"],
                "title": title,
                "thumbnail_url": thumbnail_url,
                "from_db": False,
            })
        except Exception as e:
            _push_error(vid_hash, str(e))

    threading.Thread(target=_run, daemon=True).start()

    return jsonify({"vid_hash": vid_hash, "status": "processing"})


@app.route("/load-file", methods=["POST"])
def load_file():
    if "file" not in request.files:
        return jsonify({"error": "No file part"}), 400
    file = request.files["file"]
    if not _allowed_file(file.filename):
        return jsonify({"error": "File type not allowed"}), 400

    ext      = file.filename.rsplit(".", 1)[-1].lower()
    raw_path = os.path.join(UPLOAD_DIR, f"upload.{ext}")
    file.save(raw_path)
    vid_hash  = make_file_hash(raw_path)

    db_record = get_video_from_db(vid_hash)
    if db_record and db_record.get("chunks"):
        return jsonify(
            {
                "vid_hash": vid_hash,
                "language": db_record["language"],
                "summary": db_record["summary"],
                "title": db_record.get("title") or "",
                "thumbnail_url": db_record.get("thumbnail_url") or "",
                "from_db": True,
            }
        )

    audio_path = os.path.join(UPLOAD_DIR, "audio.wav")
    if os.path.exists(audio_path):
        os.remove(audio_path)

    subprocess.run(
        ["ffmpeg", "-y", "-i", raw_path, "-ac", "1", "-ar", "16000", audio_path],
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
    )
    title = (file.filename or "Uploaded file").replace("\\", "/").split("/")[-1]

    _push_progress(vid_hash, "converting", 5, "Converting audio format…")

    def _run():
        try:
            result = process_audio(
                audio_path, vid_hash, youtube_url="", title=title, thumbnail_url=""
            )
            _push_done(vid_hash, {
                "vid_hash": vid_hash,
                "language": result["language"],
                "summary": result["summary"],
                "title": title,
                "thumbnail_url": "",
                "from_db": False,
            })
        except Exception as e:
            _push_error(vid_hash, str(e))

    threading.Thread(target=_run, daemon=True).start()

    return jsonify({"vid_hash": vid_hash, "status": "processing"})


@app.route("/export-chunks", methods=["POST"])
def export_chunks():
    data     = request.get_json(silent=True) or {}
    vid_hash = data.get("vid_hash", "").strip()
    if not vid_hash:
        return jsonify({"error": "No video hash provided"}), 400

    db_record = get_video_from_db(vid_hash)
    if not db_record or not db_record.get("chunks"):
        return jsonify({"error": "No chunks found for this video."}), 404

    return jsonify({"chunks": db_record["chunks"]})


@app.route("/library", methods=["GET"])
def library():
    return jsonify({"videos": list_videos_from_db()})


@app.route("/video/<vid_hash>", methods=["GET"])
def video_meta(vid_hash):
    rec = get_video_from_db(vid_hash)
    if not rec:
        return jsonify({"error": "Not found"}), 404
    return jsonify(
        {
            "vid_hash": vid_hash,
            "url": rec["url"],
            "summary": rec["summary"],          # ← full summary included
            "language": rec["language"],
            "title": rec.get("title") or "",
            "thumbnail_url": rec.get("thumbnail_url") or "",
        }
    )


@app.route("/chat", methods=["POST"])
def chat():
    data      = request.get_json(silent=True) or {}
    question  = data.get("question", "").strip()
    vid_hash  = data.get("vid_hash", "").strip()
    use_hyde  = data.get("use_hyde", True)
    debug     = bool(data.get("debug", False))

    if not question:
        return jsonify({"error": "Empty question."}), 400
    if not vid_hash:
        return jsonify({"error": "No video loaded."}), 400

    db_record = get_video_from_db(vid_hash)
    if not db_record:
        return jsonify({"error": "Video not found in database."}), 404

    q_type = classify_question(question)
    if q_type == "generic":
        ans = groq_chat("You are VidMind AI.", question)
        payload = {
            "answer": ans,
            "source": "llm",
            "youtube_url": "",
            "timestamp_label": "",
        }
        if debug:
            payload["debug"] = {
                "steps": [
                    {"step": 1, "name": "Route", "output": "generic_llm (no RAG)"},
                    {"step": 2, "name": "User question", "output": question},
                    {"step": 3, "name": "LLM response", "output": ans},
                ]
            }
        return jsonify(payload)
    if q_type == "summary":
        user_block = f"Summary:\n{db_record['summary']}\n\nQ: {question}"
        ans = groq_chat(
            "Answer using ONLY the summary provided.",
            user_block,
        )
        payload = {
            "answer": ans,
            "source": "summary",
            "youtube_url": "",
            "timestamp_label": "",
        }
        if debug:
            payload["debug"] = {
                "steps": [
                    {"step": 1, "name": "Route", "output": "summary_only (no transcript RAG)"},
                    {"step": 2, "name": "Summary context (excerpt)", "output": (db_record["summary"] or "")[:2000]},
                    {"step": 3, "name": "User question", "output": question},
                    {"step": 4, "name": "LLM response", "output": ans},
                ]
            }
        return jsonify(payload)

    out = answer_via_rag(
        question,
        vid_hash,
        db_record["url"],
        db_record.get("summary") or "",
        db_record.get("chunks") or [],
        use_hyde=use_hyde,
        debug=debug,
    )
    if out.get("error"):
        return jsonify(out), 400
    return jsonify(out)


@app.route("/<path:filename>")
def serve_static(filename):
    return send_from_directory(FRONTEND_DIR, filename)


# ══════════════════════════════════════════
# START
# ══════════════════════════════════════════
if __name__ == "__main__":
    ngrok_token = os.environ.get("NGROK_AUTH_TOKEN", "")
    if ngrok_token:
        ngrok.set_auth_token(ngrok_token)
        public_url = ngrok.connect(8000)
        print(f"\n✅ Website live at: {public_url}\n")
    else:
        print("\n✅ Flask: http://127.0.0.1:8000 (set NGROK_AUTH_TOKEN to expose via ngrok)\n")
    app.run(port=8000, threaded=True)

Loading models in parallel...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-large-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Warming up Whisper...
Whisper warm-up done.
Connecting to Pinecone...
✅ Index 'video-transcript-index-v7' already exists.
🚀 All models and databases ready!

✅ Website live at: NgrokTunnel: "https://everyone-apple-spoof.ngrok-free.dev" -> "http://localhost:8000"

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:8000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [13/Apr/2026 11:39:56] "GET / HTTP/1.1" 304 -
INFO:werkzeug:127.0.0.1 - - [13/Apr/2026 11:39:57] "GET /style.css HTTP/1.1" 304 -
INFO:werkzeug:127.0.0.1 - - [13/Apr/2026 11:39:57] "GET /app.js HTTP/1.1" 304 -
INFO:werkzeug:127.0.0.1 - - [13/Apr/2026 11:39:58] "GET /library HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [13/Apr/2026 11:40:02] "GET /home.html HTTP/1.1" 304 -
INFO:werkzeug:127.0.0.1 - - [13/Apr/2026 11:40:04] "GET /app.js HTTP/1.1" 304 -
INFO:werkzeug:127.0.0.1 - - [13/Apr/2026 11:40:04] "GET /style.css HTTP/1.1" 304 -
INFO:werkzeug:127.0.0.1 - - [13/Apr/2026 11:40:05] "GET /library HTTP/1.1" 200 -
ERROR: [youtube] uNBu5eSrtnA: Sign in to confirm you’re not a bot. Use --cookies-from-browser or --cookies for the authentication. See  https://github.